# Анализ лояльности пользователей Яндекс Афиши

In [ ]:
#Установка зависимостей
!pip install -r requirements.txt

# 1. Загрузка данных и их предобработка

## Задача 1.1  

Требуется написать SQL-запрос, выгружающий в датафрейм pandas необходимые данные из базы данных `data-analyst-afisha`:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа;
- `order_ts` — дата и время создания заказа;
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

In [ ]:
# Импорты
from dotenv import load_dotenv
import os

from sqlalchemy import create_engine

import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Глобальная конфигурация
mpl.rcParams['figure.constrained_layout.use'] = True

In [ ]:
# Чтение .env файла
load_dotenv(dotenv_path='.env')

In [ ]:
# Подключение к бд
engine = create_engine(os.getenv('DB_CONNECTION_STRING')) 

In [ ]:
# Получение данных
query = '''
SELECT
  user_id,
  device_type_canonical,
  order_id,
  created_dt_msk AS order_dt,
  created_ts_msk AS order_ts,
  currency_code,
  revenue,
  tickets_count,
  created_dt_msk::date - LAG(created_dt_msk::date) OVER (
    PARTITION BY user_id
    ORDER BY created_dt_msk
  ) AS days_since_prev,
  event_id,
  service_name,
  event_type_main,
  r.region_name,
  c.city_name
FROM afisha.purchases
LEFT JOIN afisha.events e USING (event_id) 
LEFT JOIN afisha.city c USING (city_id)
LEFT JOIN afisha.regions r USING (region_id)
WHERE device_type_canonical IN ('mobile', 'desktop') 
  AND event_type_main != 'фильм'
ORDER BY user_id
'''
df = pd.read_sql_query(query, con=engine)

In [ ]:
df


## Задача 1.2 

Изучить общую информацию о выгруженных данных. Оценить корректность выгрузки и объём полученных данных.

Предположить, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксировать основную информацию о данных в кратком промежуточном выводе.

In [ ]:
# Размерность
df.shape

In [ ]:
# Пропуски
df.isna().sum()

In [ ]:
# Проверка типов
df.info()

### Промежуточный вывод

Данные корректно получены из базы данных. Строк: 290611, колонок: 14

Пропуски найдены только в столбце `days_since_prev` - это ожидаемое поведение

Требуются следующие корректировки типов данных:
- Столбец days_since_prev привести к int
- Downcast числовых столбцов для уменьшения затрат памяти

# 2. Предобработка данных

## Задача 2.1

Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Требуется привести выручку к единой валюте — российскому рублю.

Для этого можно использовать датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохранить в новый столбец `revenue_rub`.

In [ ]:
tenge_rate = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv', parse_dates=['data'])
tenge_rate = tenge_rate.set_index('data')

In [ ]:
tenge_rate

In [ ]:
df['revenue_rub'] = df.apply(
    lambda row: 
        row['revenue'] 
        if row['currency_code'] == 'rub' 
        else row['revenue'] * tenge_rate.loc[row['order_dt']]['curs'] / tenge_rate.loc[row['order_dt']]['nominal'],
    axis=1
)

In [ ]:
df

## Задача 2.2

### 2.2.1 Проверить на пропущенные значения

In [ ]:
df.isna().sum()

Пропуски найдены только в столбце `days_since_prev` - это ожидаемое поведение

### 2.2.2 Преобразовать типы данных

In [ ]:
# Столбец days_since_prev привести к int
df['days_since_prev'] = pd.to_numeric(df['days_since_prev'])

# Downcast числовых столбцов для уменьшения затрат памяти
for col in df.columns:  
	if pd.api.types.is_integer_dtype(df[col]):  
		df[col] = pd.to_numeric(df[col], downcast='integer')  
	elif pd.api.types.is_float_dtype(df[col]):  
		df[col] = pd.to_numeric(df[col], downcast='float')

In [ ]:
df.info()

### 2.2.3 Изучить значения в ключевых столбцах

In [ ]:
df['service_name'].value_counts()

In [ ]:
df['event_type_main'].value_counts()

In [ ]:
df['region_name'].value_counts()

In [ ]:
df['city_name'].value_counts()

Не найдено категорий, которые обозначают пропуски или отсутствие данных

In [ ]:
# Проверка уникальных значений в каждом столбце
df.nunique()

Аномалий с дубликатами в отдельных столбцах не выявлено. В order_id все значения уникальны и их количество соответствует количеству строк в данных. Следовательно, явные дубликаты в данных отсутствуют

Попробуем найти неявные дубликаты по другим комбинациям столбцов

In [ ]:
duplicate_checks = [
    ['user_id', 'event_id', 'order_ts'],
    ['user_id', 'event_id', 'order_ts', 'revenue', 'tickets_count', 'currency_code']
]

for cols in duplicate_checks:
    print(cols, ':')
    print(df.duplicated(subset=cols, keep='first').sum())
    print('---')

Есть 120 дубликатов по комбинации столбцов `'user_id', 'event_id', 'order_ts'`. Можно предположить, что часть их них получилась из-за особенностей системы заказов. Например, из-за разбиения билетов по отдельным заказам или из-за обработки системой очерди заказов пачками

Поэтому уточним столбы для поиска дубликатов до набора `'user_id', 'event_id', 'order_ts', 'revenue', 'tickets_count', 'currency_code'`. Тогда получается 44 дубликата, в которых повторяется все, кроме id заказа. Эти дубликаты удалим из набора данных

В реальной задаче в подобной ситуации перед удалением стоит уточнить у отдела, работающего с бд, о природе подобных дубликатов

In [ ]:
# Удаление дубликатов
df_len = len(df)
print(f'Строк до удаления {df_len}')
df = df.drop_duplicates(subset=['user_id', 'event_id', 'order_ts', 'revenue', 'tickets_count', 'currency_code'], keep='first')
print(f'Строк после удаления {len(df)}. Удалено строк: {df_len - len(df)}')

Изучим разброс выручки заказов

In [ ]:
df['revenue_rub'].describe()

In [ ]:
# Построение графиков с разбросом выручки заказа
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

fig.suptitle('Разброс выручки заказа')

sns.histplot(df['revenue_rub'], ax=axes[0])
axes[0].set(xlabel='Выручка', ylabel='Количество')
sns.boxplot(df['revenue_rub'], ax=axes[1])
axes[1].set(xlabel='', ylabel='Выручка')

plt.show()

In [ ]:
# Считаем маску для удаления из данных строк с `revenue_rub` больше 99 перцентиля + удаление отрицательных значений
# Применим после построения графиков
df_revenue_rub_99_mask = (
    (df['revenue_rub'] < df['revenue_rub'].quantile(.99))
    & (df['revenue_rub'] >= 0)
)

In [ ]:
revenue_rub_more_99_percentile = (~df_revenue_rub_99_mask).sum()

print(
    f"Всего заказов: {len(df)}, "
    f"заказов с выручкой ≥ 99-го перцентиля: {revenue_rub_more_99_percentile}, "
    f"доля: {revenue_rub_more_99_percentile / len(df):.2%}"
)

In [ ]:
# Построение графиков с разбросом выручки заказа без выбросов
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

fig.suptitle('Разброс выручки заказа без выбросов')

revenue_rub_99 = df[df_revenue_rub_99_mask]['revenue_rub']

sns.histplot(revenue_rub_99, ax=axes[0])
axes[0].set(xlabel='Выручка', ylabel='Количество')
sns.boxplot(revenue_rub_99, ax=axes[1])
axes[1].set(xlabel='', ylabel='Выручка')

plt.show()

Даже без учета выбросов графики показывают правостороннюю асимметрию распределения выручки.

- Большинство заказов имеют небольшую стоимость (примерно 200–300 рублей).
- При этом наблюдается небольшой процент заказов с высокой выручкой (более 2000 рублей), формирующий длинный правый хвост распределения.

Такое распределение является типичным для данных о покупках билетов, где стоимость заказа может существенно различаться в зависимости от события и количества билетов.

Для более наглядного анализа формы распределения построим график распределения логарифма выручки.

In [ ]:
#Построение графика логарифма выручки
fig, ax = plt.subplots(figsize=(12, 6))

fig.suptitle('Разброс логарифма выручки заказа без выбросов')

revenue_rub_99_log = np.log1p(revenue_rub_99)

sns.histplot(revenue_rub_99_log, ax=ax)
ax.set(xlabel='Логарифм выручки', ylabel='Количество')

plt.show()

После логарифмирования стало заметно, что значительное количество заказов имеет нулевую выручку (log1p(0) = 0), что проявляется отдельным высоким столбцом в точке 0.

Теперь проведем анализ количества билетов в заказе (`tickets_count`).

In [ ]:
df['tickets_count'].describe()

In [ ]:
# Построение графиков с разбросом количества билетов в заказе
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

fig.suptitle('Разброс количества билетов')

sns.barplot(df.groupby('tickets_count').size(), ax=axes[0])
axes[0].set(xlabel='Количество билетов', ylabel='Количество заказов')
sns.boxplot(df['tickets_count'], ax=axes[1])
axes[1].set(xlabel='', ylabel='Количество билетов')

plt.show()

Построению графиков и анализу данных мешают заказы с большим количеством билетов. Посмотрим на количество таких заказов

In [ ]:
df.groupby('tickets_count').size().sort_index(ascending=False)

In [ ]:
# Количество заказов с количеством билетов больше 99 перцентилю
tickets_count_amount_more_99_percentile = (df['tickets_count'] >= df['tickets_count'].quantile(.99)).sum()

print(
    f"Всего заказов: {len(df)}, "
    f"заказов с количеством билетов ≥ 99-го перцентиля: {tickets_count_amount_more_99_percentile}, "
    f"доля: {tickets_count_amount_more_99_percentile / len(df):.2%}"
)


Заказы с количеством билетов выше 99-го перцентиля составляют около 1.54% от всех заказов, что соответствует ожидаемой доле для верхнего хвоста распределения. Эти наблюдения представляют редкие случаи крупных покупок билетов.

Уберем также данные по 99 перцентилю. Отрицательного количества билетов в заказах не выявлено

In [ ]:
# Считаем маску для удаления из данных строк с `tickets_count` больше 99 перцентиля
# Применим после построения графиков
df_tickets_count_99_mask = (df['tickets_count'] < df['tickets_count'].quantile(.99))

In [ ]:
# Построение графиков с разбросом количества билетов в заказе без выбросов
fig, axes = plt.subplots(figsize=(6, 6))

fig.suptitle('Разброс количества билетов без выбросов')

df_tickets_count_99 = df[df_tickets_count_99_mask]

sns.barplot(df_tickets_count_99.groupby('tickets_count').size(), ax=axes)
axes.set(xlabel='Количество билетов', ylabel='Количество заказов')

plt.show()

Большинство заказов — 2–3 билета. С ростом количества билетов число заказов уменьшается; покупки 5 билетов встречаются редко. Это указывает на то, что пользователи чаще всего покупают билеты для себя или небольшой компании.

Дополнительно проанализируем корреляцию между количеством билетов в заказе и его итоговой стоимостью:

In [ ]:
p = sns.jointplot(
    data=df,
    x='tickets_count',
    y='revenue_rub',
    kind='scatter',
    marginal_kws=dict(bins=30, fill=True),
)
p.set_axis_labels('Количество билетов', 'Выручка')
plt.show()

Заметной зависимости между количеством билетов в заказе и итоговой выручкой не наблюдается. Большинство заказов содержит небольшое количество билетов (1–4) с относительно низкой стоимостью.

Высокая выручка в отдельных заказах обусловлена, вероятно, высокой ценой билетов на конкретные мероприятия, а не большим количеством билетов.

При этом заказы с наибольшим количеством билетов имеют нулевую или почти нулевую стоимость, что может указывать на бесплатные мероприятия, на которые оформлялись групповые заказы.

Очистим данные от выбросов:

In [ ]:
df_len_before_cleaning = len(df)
print(f'Количество строк до удаления: {df_len_before_cleaning}')

df = df[df_revenue_rub_99_mask & df_tickets_count_99_mask]

df_len_after_cleaning = len(df)

print(
    f'Количество строк после удаления: {df_len_after_cleaning} '
    f'(удалено: {df_len_before_cleaning - df_len_after_cleaning}, '
    f'доля удалённых: {(df_len_before_cleaning - df_len_after_cleaning) / df_len_before_cleaning:.2%})'
)

### Промежуточный вывод

1. Добавлен столбец `revenue_rub` - стоимость заказа, приведённая к российскому рублю.
2. Пропуски найдены только в столбце `days_since_prev` - это ожидаемое поведение
3. Типы данных преобразованы к корректным и более оптимальным
4. Удалено 44 строки с неявными дубликатами
5. Явной сильной зависимости между стоимостью заказа и количеством билетов не выявлено
6. Удалено 4470 строк с выбросами по стоимости заказа или количеству билетов.

# 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

## Задача 3.1

Построить профиль пользователя — для каждого пользователя найти:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используя поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

In [ ]:
df

In [ ]:
def make_profile(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values('order_ts')
    return (
        df.groupby('user_id').agg(
            order_dt_first=('order_dt', 'first'),
            order_dt_last=('order_dt', 'last'),
            first_device=('device_type_canonical', 'first'),
            first_region=('region_name', 'first'),
            first_service_name=('service_name', 'first'),
            first_event_type=('event_type_main', 'first'),
            orders_amount=('order_id', 'size'),
            avg_revenue_rub=('revenue_rub', 'mean'),
            mean_tickets_count=('tickets_count', 'mean'),
            mean_days_between_orders=('days_since_prev', 'mean'),
            is_two=('order_id', lambda orders: len(orders) >= 2),
            is_five=('order_id', lambda orders: len(orders) >= 5),
        )
    )

make_profile(df)